# Build Station Data — Full Range (2021-06-29 → 2026-04-30)

Builds `vd_station_1m` and `vd_station_5m` from existing raw VD parquet files.

**Resumable** — already-built days are automatically skipped. If the session disconnects, just re-run from Cell 1.

**Estimated runtime** — ~8–14 h on Colab for ~1,300 new days. Free-tier sessions may disconnect; re-running is safe.

### Run order
1. Cell 1 — Mount Drive + verify
2. Cell 2 — Install uv
3. Cell 3 — Configure
4. Cell 4 — Dry run *(optional)*
5. **Cell 5 — Build** *(main, long-running)*
6. Cell 6 — Verify output files
7. Cell 7 — Post-build Drive sync check

In [ ]:
# ── Cell 1: Mount Drive + verify ──────────────────────────────────────────────
# Uses force_remount=True to avoid stale-cache mounts.
# Writes and reads back a sentinel file to confirm Drive is actually writable
# (not just a locally cached view that won't persist after session end).

import os, sys, time, json, pathlib, subprocess

from google.colab import drive
print('Mounting Google Drive (force_remount=True)...')
drive.mount('/content/drive', force_remount=True)

DRIVE_ROOT   = pathlib.Path('/content/drive/MyDrive')
PROJECT_ROOT = DRIVE_ROOT / '陽交大高速公路計畫' / 'main'
DATA_ROOT    = PROJECT_ROOT / 'data'
SCRIPT       = PROJECT_ROOT / 'scripts' / 'data' / 'build_station_vd_drive.py'

# ── 1a. Verify critical paths exist ───────────────────────────────────────────
for p, label in [
    (PROJECT_ROOT,                       'project root'),
    (SCRIPT,                             'build script'),
    (DATA_ROOT / 'vd_raw_lane_1m_full',  'raw VD data'),
]:
    if not p.exists():
        raise FileNotFoundError(f'{label} not found at: {p}')
    print(f'  ✓ {label}')

# ── 1b. Sentinel write/read — confirms Drive is not just a stale local cache ──
sentinel = PROJECT_ROOT / '.colab_sentinel'
ts = str(time.monotonic())
tag = f'ok:{ts}'
try:
    sentinel.write_text(tag)
    time.sleep(2)
    rb = sentinel.read_text()
    if rb != tag:
        raise AssertionError(f'wrote {tag!r} but read back {rb!r}')
    sentinel.unlink()
    print('✓ Drive write/read sentinel OK (real Drive, not stale cache)')
except Exception as exc:
    try:
        sentinel.unlink()
    except Exception:
        pass
    raise RuntimeError(
        f'Drive sentinel check FAILED: {exc}. '
        'Try: Runtime > Factory reset runtime, then re-run this cell.'
    ) from exc

# ── 1c. Show Drive free space ──────────────────────────────────────────────────
stat = os.statvfs('/content/drive')
free_gb  = stat.f_bavail * stat.f_frsize / 1e9
total_gb = stat.f_blocks * stat.f_frsize / 1e9
print(f'  Drive: {free_gb:.1f} GB free / {total_gb:.1f} GB total')
print('Ready.')

In [ ]:
# ── Cell 2: Install uv ────────────────────────────────────────────────────────
result = subprocess.run(['pip', 'install', 'uv', '-q'], capture_output=True, text=True)
if result.returncode != 0:
    print('pip stderr:', result.stderr)
    raise RuntimeError('pip install uv failed')

r = subprocess.run(['uv', '--version'], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError('uv not available after install — check PATH')
print(r.stdout.strip())

In [ ]:
# ── Cell 3: Configuration + pre-build snapshot ────────────────────────────────
# Adjust START_DATE / END_DATE here if you want to build a subset first.

START_DATE = '2023-08-06'
END_DATE   = '2026-04-30'

def count_parquet(root, pattern):
    p = pathlib.Path(root)
    if not p.exists():
        return 0
    return sum(1 for _ in p.rglob(pattern))

raw_n = count_parquet(DATA_ROOT / 'vd_raw_lane_1m_full', 'vd_*.parquet')
st5_n = count_parquet(DATA_ROOT / 'vd_station_5m',       'vd_station_*.parquet')
st1_n = count_parquet(DATA_ROOT / 'vd_station_1m',       'vd_station_*.parquet')

print(f'Date range   : {START_DATE}  →  {END_DATE}')
print(f'Raw VD files : {raw_n:,}   (each = 1 source day)')
print(f'Station 5m   : {st5_n:,}   existing  → will be skipped')
print(f'Station 1m   : {st1_n:,}   existing  → will be skipped')
print(f'To build     : ~{max(raw_n - st5_n, 0):,} new days')

if raw_n == 0:
    raise RuntimeError('No raw VD parquet files found — check that DATA_ROOT is correct and Drive is mounted.')

In [ ]:
# ── Cell 4 (optional): Dry run — prints planned paths without writing anything ─
r = subprocess.run(
    [
        'uv', 'run', str(SCRIPT),
        '--start-date', START_DATE,
        '--end-date',   END_DATE,
        '--data-root',  str(DATA_ROOT),
        '--dry-run',
    ],
    capture_output=True, text=True, errors='replace', cwd=str(PROJECT_ROOT),
)
out = r.stdout
print(out[-4000:] if len(out) > 4000 else out)
if r.returncode != 0:
    print('STDERR:', r.stderr[:2000])

In [ ]:
# ── Cell 5: Run build ─────────────────────────────────────────────────────────
# Already-built days are skipped (status=skipped_existing).
# If the session disconnects, re-run from Cell 1 — progress is preserved.

cmd = [
    'uv', 'run', str(SCRIPT),
    '--start-date', START_DATE,
    '--end-date',   END_DATE,
    '--data-root',  str(DATA_ROOT),
]
print('Command:', ' '.join(str(c) for c in cmd))
print('=' * 70)
t0 = time.time()

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=str(PROJECT_ROOT),
    errors='replace',
)

try:
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
except KeyboardInterrupt:
    proc.terminate()
    proc.wait()
    print()
    print('[INTERRUPTED — progress saved; re-run this cell to resume]')

elapsed = time.time() - t0
print()
print('=' * 70)
print(f'Elapsed: {elapsed / 60:.1f} min  |  Exit code: {proc.returncode}')
if proc.returncode not in (0, None):
    print('WARNING: non-zero exit — check the log above for error details.')

In [ ]:
# ── Cell 6: Verify output files ───────────────────────────────────────────────
# Counts files, spot-reads 5 random station_5m files, shows build report.

import random
try:
    import pyarrow.parquet as pq
except ImportError:
    subprocess.run(['pip', 'install', 'pyarrow', '-q'], check=True)
    import pyarrow.parquet as pq

st5_root = DATA_ROOT / 'vd_station_5m'
st1_root = DATA_ROOT / 'vd_station_1m'

files_5m = sorted(st5_root.rglob('vd_station_*.parquet')) if st5_root.exists() else []
files_1m = sorted(st1_root.rglob('vd_station_*.parquet')) if st1_root.exists() else []

print(f'Station 5m : {len(files_5m):,} files')
if files_5m:
    print(f'  First : {files_5m[0].name}')
    print(f'  Last  : {files_5m[-1].name}')

print(f'Station 1m : {len(files_1m):,} files')
if files_1m:
    print(f'  First : {files_1m[0].name}')
    print(f'  Last  : {files_1m[-1].name}')

# ── Spot-read 5 random station_5m files ───────────────────────────────────────
problems = []
if files_5m:
    print()
    print('Spot-check (5 random station_5m files):')
    sample = random.sample(files_5m, min(5, len(files_5m)))
    for f in sample:
        try:
            sz_kb = f.stat().st_size / 1024
            nrows = pq.ParquetFile(str(f)).metadata.num_rows
            ok    = sz_kb > 1 and nrows > 0
        except Exception as exc:
            sz_kb, nrows, ok = 0, f'READ ERROR: {exc}', False
        icon = '✓' if ok else '✗'
        print(f'  {icon} {f.name}: {sz_kb:.0f} KB, {nrows} rows')
        if not ok:
            problems.append(str(f))

    if problems:
        print(f'  ⚠ {len(problems)} file(s) have issues — see build log for details')
    else:
        print('  ✓ All spot-checked files look valid')

# ── Build report ──────────────────────────────────────────────────────────────
report_path = DATA_ROOT / 'metadata' / 'station_build_report.json'
if report_path.exists():
    rpt = json.loads(report_path.read_text())
    print()
    print('Build report:')
    print(f'  Statuses : {rpt.get("statuses", {})}')
    totals = rpt.get('totals', {})
    for k in ('station_1m_rows', 'station_5m_rows', 'station_1m_size_mb', 'station_5m_size_mb'):
        if k in totals:
            v = totals[k]
            print(f'  {k} : {v:,.0f}')
else:
    print('(station_build_report.json not found yet)')

In [ ]:
# ── Cell 7: Post-build Drive sync check ───────────────────────────────────────
# Writes and reads back a sentinel AFTER the build.
# If this passes, Drive is still syncing correctly.
# If it fails, some output files may not have been pushed to cloud Drive yet
# (wait a few minutes and retry this cell before closing the session).

sentinel2 = PROJECT_ROOT / '.colab_done'
ts2 = str(time.monotonic())
tag2 = f'done:{ts2}:{len(files_5m)}x5m:{len(files_1m)}x1m'

try:
    sentinel2.write_text(tag2)
    time.sleep(3)
    rb2 = sentinel2.read_text()
    if rb2 != tag2:
        raise AssertionError(f'wrote {tag2!r}, read back {rb2!r}')
    sentinel2.unlink()
    print('✓ Post-build Drive write/read verified')
    print(f'✓ {len(files_5m):,} station_5m files on Drive')
    print(f'✓ {len(files_1m):,} station_1m files on Drive')
    print()
    print('All done!')
except Exception as exc:
    try:
        sentinel2.unlink()
    except Exception:
        pass
    print(f'✗ Post-build Drive check FAILED: {exc}')
    print('  Wait 2–3 minutes (Drive may still be flushing writes), then re-run this cell.')
    print('  If it keeps failing, check Drive storage quota and connectivity.')